In [7]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

from google.colab import drive
drive.mount('/content/drive')

RANDOM_STATE = 42
TARGET_COL = "PlacementStatus"
DATA_PATH = "/content/drive/MyDrive/colab/placement_predict_50k.csv"

Mounted at /content/drive


In [8]:
# --------------------------------------------------------------------------
# 1. Load + preprocess (plain top-level script code, no function wrapper)
# --------------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)

# Drop the injected-anomaly flag; it's a data-quality marker, not a real feature
if "IsAnomaly" in df.columns:
    df = df.drop(columns=["IsAnomaly"])

y = df[TARGET_COL].astype(int)
X = df.drop(columns=[TARGET_COL])

cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()

# Label-encode categoricals (fine for tree-based boosters; no dummy blow-up)
encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c].astype(str))
    encoders[c] = le

# Impute missing numerics (median) — AdaBoost's DecisionTree base estimator
# can't handle NaN natively, unlike XGBoost, so both models get the same clean input
imputer = SimpleImputer(strategy="median")
X[num_cols] = imputer.fit_transform(X[num_cols])

# Scale numerics — helps AdaBoost's (default) shallow-tree base estimator converge cleanly
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

In [9]:
# --------------------------------------------------------------------------
# 2. Train / validation / test split (also plain top-level code)
# --------------------------------------------------------------------------
VAL_SIZE = 0.15
TEST_SIZE = 0.15

# First carve off test, then split remainder into train/val
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_ratio,
    stratify=y_train_val, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

Train: (34999, 19) | Val: (7501, 19) | Test: (7500, 19)


## Week 7

In [10]:
# ---- Model 1: AdaBoost ------------------------------------------------
# AdaBoost re-weights training samples each round: misclassified samples
# get higher weight so the next weak learner focuses on them.
ada_base = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE)
ada = AdaBoostClassifier(
    estimator=ada_base,
    n_estimators=200,
    learning_rate=0.5,
    random_state=RANDOM_STATE,
)

t0 = time.time()
ada.fit(X_train, y_train)
ada_fit_time = time.time() - t0

ada_val_pred = ada.predict(X_val)
ada_val_proba = ada.predict_proba(X_val)[:, 1]

ada_results = {
    "model": "AdaBoost",
    "val_accuracy": accuracy_score(y_val, ada_val_pred),
    "val_f1": f1_score(y_val, ada_val_pred),
    "val_roc_auc": roc_auc_score(y_val, ada_val_proba),
    "best_n_estimators": ada.n_estimators,   # AdaBoost has no early stopping
    "fit_time_sec": round(ada_fit_time, 2),
}
ada_results

{'model': 'AdaBoost',
 'val_accuracy': 0.796293827489668,
 'val_f1': 0.7809633027522935,
 'val_roc_auc': np.float64(0.8801620529375686),
 'best_n_estimators': 200,
 'fit_time_sec': 11.74}

## Week 8

In [11]:
# ---- Model 2: XGBoost with early stopping -----------------------------
# early_stopping_rounds lives on the constructor in xgboost>=2.0; fit()
# is given eval_set and stops once val logloss hasn't improved in N rounds.
xgb = XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

t0 = time.time()
xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
xgb_fit_time = time.time() - t0

xgb_val_pred = xgb.predict(X_val)
xgb_val_proba = xgb.predict_proba(X_val)[:, 1]

xgb_results = {
    "model": "XGBoost",
    "val_accuracy": accuracy_score(y_val, xgb_val_pred),
    "val_f1": f1_score(y_val, xgb_val_pred),
    "val_roc_auc": roc_auc_score(y_val, xgb_val_proba),
    "best_n_estimators": xgb.best_iteration + 1,  # rounds actually used before stopping
    "fit_time_sec": round(xgb_fit_time, 2),
}
xgb_results

{'model': 'XGBoost',
 'val_accuracy': 0.7958938808158912,
 'val_f1': 0.7827444302540087,
 'val_roc_auc': np.float64(0.8826375192126859),
 'best_n_estimators': 135,
 'fit_time_sec': 2.42}

In [12]:
# --------------------------------------------------------------------------
# 4. Combine results into leaderboard
# --------------------------------------------------------------------------
leaderboard = pd.DataFrame([ada_results, xgb_results]).sort_values(
    "val_accuracy", ascending=False
).reset_index(drop=True)

print("\nValidation leaderboard (sorted by val_accuracy):")
print(leaderboard.to_string(index=False))

leaderboard.to_csv("boosting_benchmark_results.csv", index=False)
print("\nSaved results to boosting_benchmark_results.csv")


Validation leaderboard (sorted by val_accuracy):
   model  val_accuracy   val_f1  val_roc_auc  best_n_estimators  fit_time_sec
AdaBoost      0.796294 0.780963     0.880162                200         11.74
 XGBoost      0.795894 0.782744     0.882638                135          2.42

Saved results to boosting_benchmark_results.csv
